# Análisis de Regresión — Regresión Logística Ordinal
## TFM: Relación entre el uso docente y la valoración del profesorado de una plataforma de aprendizaje en línea

Modela la relación entre el uso de grupos funcionales del LMS y la categoría NPS del docente
mediante **regresión logística ordinal** (modelo de odds proporcionales).

**Variable dependiente:** categoría NPS — detractor (0) / indiferente (1) / promotor (2)
**Variables independientes:** tasas e intensidades de uso por 11 grupos funcionales

**Prerequisito:** ejecutar `00_base_teacher_events.sql` → `01a_gold_table_features_groups.sql`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.family': 'serif', 'font.size': 11, 'figure.dpi': 150})
FIGURES_DIR = os.path.join(os.path.dirname(os.getcwd()), 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f'Figuras en: {FIGURES_DIR}')


In [ ]:
# ---------------------------------------------------------------------------
# DATOS: acceso requiere infraestructura autorizada (Databricks).
# Los notebooks documentan el codigo analitico con fines de transparencia.
#
# Esquema esperado - gold_nps_features_groups (una fila por docente-respuesta):
#   nps_score              int     Puntuacion NPS (0-10)
#   nps_category           str     'detractor' | 'passive' | 'promoter'
#   n_sessions             int     Numero de sesiones en ventana 90 dias
#   clicks_per_session     float   Media de clics por sesion
#   n_distinct_features    int     Numero de funcionalidades distintas usadas
#   total_feature_clicks   int     Total de clics en la ventana
#   rate_{group}           float   Clics del grupo / sesiones totales
#   intensity_{group}      float   Clics del grupo / sesiones con ese grupo
#   (grupos: grading, modules, discussions, assignments, quizzes,
#    navigation, announcements, people, analytics, admin, otros)
# ---------------------------------------------------------------------------
raise NotImplementedError('Acceso a datos requiere infraestructura autorizada.')


## 1. Definición y preparación de variables

**Tres familias de predictores:**
- `rate_*` (11 cols): clics del grupo / sesiones totales → uso normalizado por volumen
- `intensity_*` (11 cols): clics del grupo / sesiones con ese grupo → intensidad condicional
- `behav_cols`: indicadores de comportamiento transversal (`clicks_per_session`, `n_sessions`,
  `n_distinct_features`) + variable de escala (`total_feature_clicks`)

**Variable dependiente:** `nps_ord` (0 = detractor, 1 = indiferente, 2 = promotor)

In [0]:
# Variable dependiente ordinal
cat_map = {'detractor': 0, 'passive': 1, 'promoter': 2}
df['nps_ord'] = df['nps_category'].map(cat_map)

# Familia 1: tasa normalizada por sesiones totales
rate_cols   = [c for c in df.columns if c.startswith('rate_')]
# Familia 2: intensidad condicional (clics / sesiones con uso de ese grupo)
intens_cols = [c for c in df.columns if c.startswith('intensity_')]
# Familia 3: comportamiento transversal
behav_cols  = ['clicks_per_session', 'n_sessions', 'n_distinct_features']
# Variable de escala para Modelo D
scale_col   = 'total_feature_clicks'

y       = df['nps_ord']
id_cols = ['userhasheduuid', 'nps_score', 'nps_category', 'nps_ord']

print(f'N docentes: {len(df):,}')
print(f'Distribución NPS: {y.value_counts().sort_index().to_dict()}')
print(f'Variables rate_      ({len(rate_cols)}): {rate_cols}')
print(f'Variables intensity_ ({len(intens_cols)}): {intens_cols}')
print(f'Variables behav      ({len(behav_cols)}): {behav_cols}')

## 2. Ingeniería de características

Winsorización al p99 (basada en datos de entrenamiento) + StandardScaler.
Se documentan las variables dispersas (alto % de ceros) y su tratamiento con `fillna(0)`.

In [0]:
def prepare_X(df_in, cols, winsorize_p=0.99):
    """Winsorización al p99, fillna(0) y z-score sobre los datos proporcionados."""
    d = df_in[cols].copy()
    for col in cols:
        cap = d[col].quantile(winsorize_p)
        d[col] = d[col].clip(upper=cap)
    d = d.fillna(0)
    scaler = StandardScaler()
    return pd.DataFrame(scaler.fit_transform(d), columns=cols, index=df_in.index)

def prepare_X_pair(df_train, df_test, cols, winsorize_p=0.99):
    """Winsorización y escalado basados SOLO en train; aplica la misma transformación a test."""
    d_tr = df_train[cols].copy().fillna(0)
    d_te = df_test[cols].copy().fillna(0)
    for col in cols:
        cap = d_tr[col].quantile(winsorize_p)
        d_tr[col] = d_tr[col].clip(upper=cap)
        d_te[col] = d_te[col].clip(upper=cap)
    scaler = StandardScaler().fit(d_tr)
    return (pd.DataFrame(scaler.transform(d_tr), columns=cols, index=df_train.index),
            pd.DataFrame(scaler.transform(d_te), columns=cols, index=df_test.index))

# Preparación sobre conjunto completo (para modelos finales y reportes)
X_rate   = prepare_X(df, rate_cols)
X_intens = prepare_X(df, intens_cols)

# Tabla resumen: % ceros, media y std de las variables originales
summary_rows = []
for col in rate_cols + intens_cols:
    raw = df[col].fillna(0)
    summary_rows.append({
        'variable':   col,
        'pct_zeros':  round((raw == 0).mean() * 100, 1),
        'media_raw':  round(raw.mean(), 5),
        'std_raw':    round(raw.std(), 5),
        'p99':        round(raw.quantile(0.99), 5),
    })
feat_summary = pd.DataFrame(summary_rows)
print('=== Variables con > 30% ceros (dispersas) ===')
print(feat_summary[feat_summary['pct_zeros'] > 30]
      [['variable', 'pct_zeros', 'media_raw', 'p99']].to_string(index=False))
print(f'\nForma final — X_rate: {X_rate.shape}  |  X_intens: {X_intens.shape}')

## 3. Análisis de multicolinealidad (VIF)

Se calculan los Factores de Inflación de la Varianza (VIF) para detectar multicolinealidad.
Variables con VIF > 10 se excluyen de los modelos correspondientes.

In [0]:
def check_vif(X):
    X_vif = sm.add_constant(X)
    vif = pd.DataFrame({
        'feature': X_vif.columns,
        'VIF':     [variance_inflation_factor(X_vif.values, i)
                    for i in range(X_vif.shape[1])]
    }).iloc[1:].sort_values('VIF', ascending=False).reset_index(drop=True)
    return vif

vif_rate   = check_vif(X_rate)
vif_intens = check_vif(X_intens)

print('=== VIF — Tasa simple (rate_) ===')
print(vif_rate.to_string(index=False))
print('\n=== VIF — Intensidad condicional (intensity_) ===')
print(vif_intens.to_string(index=False))

high_vif_rate   = vif_rate[vif_rate['VIF'] > 10]['feature'].tolist()
high_vif_intens = vif_intens[vif_intens['VIF'] > 10]['feature'].tolist()

print(f'\nrate_      — excluidas por VIF > 10: {high_vif_rate   or "ninguna"}')
print(f'intensity_ — excluidas por VIF > 10: {high_vif_intens or "ninguna"}')

## 4. Train/test split y validación cruzada

Split 80/20 estratificado por `nps_category`. El modelo A (rate_) se ajusta en el conjunto
de entrenamiento y se evalúa en el de prueba para estimar la capacidad de generalización.

La validación cruzada 5-fold se implementa manualmente: statsmodels `OrderedModel` no es
compatible con la interfaz de `sklearn` CV.

In [0]:
# Split 80/20 estratificado
df_train, df_test = train_test_split(
    df, test_size=0.2, stratify=df['nps_category'], random_state=42
)
y_train = df_train['nps_ord'].reset_index(drop=True)
y_test  = df_test['nps_ord'].reset_index(drop=True)

# Preparar X_rate para train/test con transformación basada en train
X_rate_train, X_rate_test = prepare_X_pair(df_train, df_test, rate_cols)

# Eliminar variables con VIF alto
cols_rate_clean     = [c for c in rate_cols   if c not in high_vif_rate]
X_rate_train_c      = X_rate_train[cols_rate_clean].reset_index(drop=True)
X_rate_test_c       = X_rate_test[cols_rate_clean].reset_index(drop=True)

# Ajuste del Modelo A en train
model_train  = OrderedModel(endog=y_train, exog=X_rate_train_c, distr='logit')
result_train = model_train.fit(method='bfgs', disp=False)

# Evaluación en test
probs_test  = np.asarray(result_train.predict(exog=X_rate_test_c, which='prob'))
y_pred_test = np.argmax(probs_test, axis=1)

print(f'Train: {len(df_train):,}  |  Test: {len(df_test):,}')
print(f'\nAccuracy en test (Modelo A): {accuracy_score(y_test, y_pred_test):.4f}')
baseline = (y_test == y_test.value_counts().idxmax()).mean()
print(f'Baseline (clase mayoritaria): {baseline:.4f}')
print('\nClassification report (conjunto de prueba):')
print(classification_report(y_test, y_pred_test,
      target_names=['detractor', 'passive', 'promoter']))

In [0]:
# Validación cruzada 5-fold manual — Modelo A (rate_)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_accuracies = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(df, df['nps_category']), 1):
    df_cv_tr  = df.iloc[tr_idx]
    df_cv_val = df.iloc[val_idx]
    y_cv_tr   = df_cv_tr['nps_ord'].reset_index(drop=True)
    y_cv_val  = df_cv_val['nps_ord'].reset_index(drop=True)

    X_cv_tr, X_cv_val = prepare_X_pair(df_cv_tr, df_cv_val, rate_cols)
    X_cv_tr_c  = X_cv_tr[cols_rate_clean].reset_index(drop=True)
    X_cv_val_c = X_cv_val[cols_rate_clean].reset_index(drop=True)

    try:
        m = OrderedModel(endog=y_cv_tr, exog=X_cv_tr_c, distr='logit')
        r = m.fit(method='bfgs', disp=False)
        probs = np.asarray(r.model.predict(r.params.to_numpy(), exog=X_cv_val_c.to_numpy()))
        preds = np.argmax(probs, axis=1)
        acc   = accuracy_score(y_cv_val, preds)
        cv_accuracies.append(acc)
        print(f'  Fold {fold}: accuracy = {acc:.4f}')
    except Exception as e:
        print(f'  Fold {fold}: error — {e}')

if cv_accuracies:
    print(f'\nCV Accuracy — media: {np.mean(cv_accuracies):.4f}  ±  {np.std(cv_accuracies):.4f}')

## 5. Modelos principales (cuatro especificaciones)

| Modelo | Predictores | Objetivo |
|--------|-------------|---------|
| **A** | `rate_*` | Modelo principal del TFM |
| **B** | `intensity_*` | Especificación alternativa |
| **C** | `rate_*` + `clicks_per_session`, `n_sessions`, `n_distinct_features` | Controla patrón de engagement |
| **D** | `rate_*` + `log(total_feature_clicks + 1)` | Robustez: control de escala |

Todos los modelos se ajustan sobre el **conjunto completo** para maximizar la precisión
en la estimación de coeficientes e índices de ajuste (AIC, BIC, McFadden R²).

In [0]:
def fit_ordinal(y_full, X, high_vif_cols, label):
    """Ajusta modelo ordinal logit eliminando variables con VIF > 10."""
    X_clean = X.drop(columns=[c for c in high_vif_cols if c in X.columns])
    model   = OrderedModel(endog=y_full, exog=X_clean, distr='logit')
    result  = model.fit(method='bfgs', disp=False)
    print(f'Modelo {label}: AIC={result.aic:.1f}, BIC={result.bic:.1f}, '
          f'McFadden R²={1 - result.llf/result.llnull:.4f}')
    return result, X_clean

# Convertir columnas con decimal.Decimal a float (artefacto de Snowflake)
for col in behav_cols + [scale_col]:
    if df[col].dtype == object:
        df[col] = df[col].astype(float)

# Preparar variables de comportamiento (conjunto completo)
X_behav = prepare_X(df, behav_cols)

# Variable log(total_feature_clicks + 1) para Modelo D
df['log_total_clicks'] = np.log1p(df[scale_col].fillna(0))
scaler_log = StandardScaler()
X_log = pd.DataFrame(
    scaler_log.fit_transform(df[['log_total_clicks']]),
    columns=['log_total_clicks'], index=df.index
)

# Ajustar los cuatro modelos
result_A, X_A = fit_ordinal(y, X_rate,   high_vif_rate,   'A — rate_')
result_B, X_B = fit_ordinal(y, X_intens, high_vif_intens, 'B — intensity_')
result_C, X_C = fit_ordinal(y, pd.concat([X_rate, X_behav], axis=1),
                             high_vif_rate, 'C — rate_ + behav')
result_D, X_D = fit_ordinal(y, pd.concat([X_rate, X_log], axis=1),
                             high_vif_rate, 'D — rate_ + log(total)')

In [0]:
# Comparativa AIC / BIC / McFadden R²
def model_metrics(result, label):
    r2 = 1 - (result.llf / result.llnull)
    return {'Modelo': label,
            'McFadden R²': round(r2, 4),
            'AIC':         round(result.aic, 2),
            'BIC':         round(result.bic, 2),
            'Log-lik':     round(result.llf, 2),
            'N vars':      result.df_model}

metrics_list = [
    model_metrics(result_A, 'A — rate_'),
    model_metrics(result_B, 'B — intensity_'),
    model_metrics(result_C, 'C — rate_ + behav'),
    model_metrics(result_D, 'D — rate_ + log(total)'),
]
df_metrics = pd.DataFrame(metrics_list)
print('=== Comparación de modelos ===')
print(df_metrics.to_string(index=False))

## 6. Coeficientes y forest plots

Tabla de OR + IC95% + p-valor para los cuatro modelos.
Forest plots comparativos: A vs C (estabilidad al controlar comportamiento) y A vs D (control de escala).

In [0]:
LABEL_MAP = {
    'rate_grading':            'Evaluación y calificación',
    'rate_modules':            'Módulos y contenido',
    'rate_discussions':        'Debates y comunicación',
    'rate_assignments':        'Tareas y entregas',
    'rate_quizzes':            'Evaluaciones y tests',
    'rate_navigation':         'Navegación',
    'rate_announcements':      'Anuncios',
    'rate_people':             'Gestión de estudiantes',
    'rate_analytics':          'Analítica e informes',
    'rate_admin':              'Administración',
    'rate_otros':              'Otros',
    'intensity_grading':       'Evaluación y calificación',
    'intensity_modules':       'Módulos y contenido',
    'intensity_discussions':   'Debates y comunicación',
    'intensity_assignments':   'Tareas y entregas',
    'intensity_quizzes':       'Evaluaciones y tests',
    'intensity_navigation':    'Navegación',
    'intensity_announcements': 'Anuncios',
    'intensity_people':        'Gestión de estudiantes',
    'intensity_analytics':     'Analítica e informes',
    'intensity_admin':         'Administración',
    'intensity_otros':         'Otros',
    'clicks_per_session':      'Clics por sesión',
    'n_sessions':              'N.º de sesiones',
    'n_distinct_features':     'Amplitud de uso (funcionalidades)',
    'log_total_clicks':        'Volumen total (log)',
}

def extract_coefs(result):
    ci = result.conf_int()
    coef_df = pd.DataFrame({
        'feature':    result.params.index,
        'coef':       result.params.values,
        'std_err':    result.bse.values,
        'z':          result.tvalues.values,
        'p_value':    result.pvalues.values,
        'OR':         np.exp(result.params.values),
        'OR_CI_low':  np.exp(ci.iloc[:, 0].values),
        'OR_CI_high': np.exp(ci.iloc[:, 1].values),
    }).round(4)
    # Excluir umbrales del modelo ordinal (contienen '/')
    coef_df = coef_df[~coef_df['feature'].str.contains('/')].copy()
    coef_df['sig'] = coef_df['p_value'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    )
    return coef_df.sort_values('coef', ascending=False).reset_index(drop=True)

coef_A = extract_coefs(result_A)
coef_B = extract_coefs(result_B)
coef_C = extract_coefs(result_C)
coef_D = extract_coefs(result_D)

cols_show = ['feature', 'OR', 'OR_CI_low', 'OR_CI_high', 'p_value', 'sig']
print('=== Modelo A — OR + IC95% ===')
print(coef_A[cols_show].to_string(index=False))
print('\n=== Modelo B — OR + IC95% ===')
print(coef_B[cols_show].to_string(index=False))

In [0]:
def forest_plot_single(coef_df, title, filename):
    """Forest plot de un único modelo (solo variables significativas)."""
    sig = coef_df[coef_df['p_value'] < 0.05].sort_values('OR')
    if sig.empty:
        print(f'Sin variables significativas en: {title}')
        return
    fig, ax = plt.subplots(figsize=(10, max(5, len(sig) * 0.55)))
    y_pos  = range(len(sig))
    colors = ['#2ca02c' if o > 1 else '#d62728' for o in sig['OR']]
    ax.barh(y_pos, sig['OR'] - 1, left=1, color=colors, alpha=0.7, height=0.6)
    ax.errorbar(sig['OR'], y_pos,
                xerr=[sig['OR'] - sig['OR_CI_low'], sig['OR_CI_high'] - sig['OR']],
                fmt='o', color='black', capsize=3, markersize=4)
    ax.axvline(1, color='gray', linestyle='--', linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([LABEL_MAP.get(f, f) for f in sig['feature']])
    ax.set_xlabel('Odds Ratio (IC 95%)')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}{filename}')
    plt.show()

def forest_plot_compare(coef_df1, coef_df2, label1, label2, title, filename):
    """Forest plot comparativo de dos modelos (predictores comunes, sig. en alguno)."""
    feats = coef_df1['feature'].tolist()
    m = (coef_df1[['feature', 'OR', 'OR_CI_low', 'OR_CI_high', 'p_value']]
         .rename(columns={'OR': 'OR1', 'OR_CI_low': 'CL1', 'OR_CI_high': 'CH1', 'p_value': 'pv1'})
         .merge(
             coef_df2[coef_df2['feature'].isin(feats)]
             [['feature', 'OR', 'OR_CI_low', 'OR_CI_high', 'p_value']]
             .rename(columns={'OR': 'OR2', 'OR_CI_low': 'CL2', 'OR_CI_high': 'CH2', 'p_value': 'pv2'}),
             on='feature', how='inner'))
    sig = m[(m['pv1'] < 0.05) | (m['pv2'] < 0.05)].sort_values('OR1')
    if sig.empty:
        print(f'Sin variables significativas en: {title}')
        return
    fig, ax = plt.subplots(figsize=(11, max(5, len(sig) * 0.65)))
    y_pos = np.arange(len(sig))
    offset, c1, c2 = 0.2, '#1f77b4', '#ff7f0e'
    ax.errorbar(sig['OR1'], y_pos + offset,
                xerr=[sig['OR1'] - sig['CL1'], sig['CH1'] - sig['OR1']],
                fmt='o', color=c1, capsize=3, markersize=5, label=label1)
    ax.errorbar(sig['OR2'], y_pos - offset,
                xerr=[sig['OR2'] - sig['CL2'], sig['CH2'] - sig['OR2']],
                fmt='s', color=c2, capsize=3, markersize=5, label=label2)
    ax.axvline(1, color='gray', linestyle='--', linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([LABEL_MAP.get(f, f) for f in sig['feature']])
    ax.set_xlabel('Odds Ratio (IC 95%)')
    ax.set_title(title)
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}{filename}')
    plt.show()

forest_plot_single(coef_A, 'Modelo A — Tasa de uso por grupo funcional\nRegresión logística ordinal',
                   '06a_forest_rate.png')
forest_plot_single(coef_B, 'Modelo B — Intensidad condicional de uso\nRegresión logística ordinal',
                   '06b_forest_intensity.png')

In [0]:
print('\n=== Modelo C — OR + IC95% ===')
print(coef_C[cols_show].to_string(index=False))
print('\n=== Modelo D — OR + IC95% ===')
print(coef_D[cols_show].to_string(index=False))

In [0]:
forest_plot_single(coef_C, 'Modelo C — Tasa de uso + indicadores de comportamiento',
                   '06a_forest_rate_plus_behavior.png')
forest_plot_single(coef_D, 'Modelo D — Tasa de uso + volumen total de interacción',
                   '06b_forest_intensity_plus_log_totals.png')

In [0]:
# Forest plot comparativo: A vs C (estabilidad al controlar comportamiento)
forest_plot_compare(
    coef_A, coef_C,
    'Modelo A (rate_)', 'Modelo C (rate_ + comportamiento)',
    'Estabilidad de coeficientes: Modelo A vs Modelo C',
    '06c_forest_A_vs_C.png'
)

# Forest plot comparativo: A vs D (estabilidad al controlar volumen total)
forest_plot_compare(
    coef_A, coef_D,
    'Modelo A (rate_)', 'Modelo D (rate_ + log total)',
    'Estabilidad de coeficientes: Modelo A vs Modelo D',
    '06d_forest_A_vs_D.png'
)

## 7. Efectos marginales (AME)

Los efectos marginales promedio (Average Marginal Effects) cuantifican el cambio esperado en
la **probabilidad de ser promotor** por un incremento de una desviación típica en cada predictor.

Se obtienen con `result.get_margeff()` de statsmodels (disponible en versión ≥ 0.13).

In [0]:
from scipy.special import expit

# AME manual para OrderedModel (get_margeff() no existe en OrderedModelResults)
# ∂P(y=2)/∂x_k = E[f(τ2 − xβ)] · β_k   (promotor)
# ∂P(y=1)/∂x_k = E[f(τ1 − xβ) − f(τ2 − xβ)] · β_k  (pasivo)
# ∂P(y=0)/∂x_k = −E[f(τ1 − xβ)] · β_k  (detractor)

k          = result_A.model.k_vars
beta       = result_A.params.values[:k]
thresh_raw = result_A.params.values[k:]

# Reconstruir umbrales reales (statsmodels: log-diferencias para garantizar orden)
thresh = np.zeros(len(thresh_raw))
thresh[0] = thresh_raw[0]
for j in range(1, len(thresh_raw)):
    thresh[j] = thresh[j-1] + np.exp(thresh_raw[j])

# PDF logística evaluada en cada umbral para cada observación
xb     = X_A.values @ beta                  # (n,)
f_cdf  = expit(thresh - xb[:, None])        # (n, 2)
f_vals = f_cdf * (1 - f_cdf)               # densidad logística (n, 2)
mean_f = f_vals.mean(axis=0)               # [E[f(τ1)], E[f(τ2)]]

ame_prom = mean_f[1] * beta
ame_pass = (mean_f[0] - mean_f[1]) * beta
ame_det  = -mean_f[0] * beta

me_df = pd.DataFrame({
    'feature':       list(result_A.model.exog_names[:k]),
    'AME_promoter':  ame_prom,
    'AME_passive':   ame_pass,
    'AME_detractor': ame_det,
}).reset_index(drop=True)
me_df['label'] = me_df['feature'].map(lambda f: LABEL_MAP.get(f, f))
me_df = me_df.sort_values('AME_promoter', ascending=False)
me_promoter = me_df  # alias para la celda de exportación

print('=== Efectos marginales promedio (AME) — P(promotor) ===')
print(me_df[['label', 'AME_promoter', 'AME_passive', 'AME_detractor']].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, max(4, len(me_df) * 0.55)))
colors = ['#2ca02c' if v > 0 else '#d62728' for v in me_df['AME_promoter']]
ax.barh(me_df['label'], me_df['AME_promoter'], color=colors, alpha=0.8)
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Efecto marginal promedio (AME) sobre P(promotor)')
ax.set_title('Efectos marginales — Modelo A\n(todos los predictores)')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}07_marginal_effects.png')
plt.show()

me_df.to_csv(f'{FIGURES_DIR}efectos_marginales_modeloA.csv', index=False)
print('Fig. 07 y CSV guardados.')

## 8. Evaluación y matrices de confusión

Las matrices de confusión se evalúan sobre el **conjunto de prueba** (20% retenido en § 4).
Se incluye un clasificador de referencia (*majority class*) para contextualizar el accuracy.

In [0]:
def confusion_report_test(result, X_test, y_test_vals, label, filename):
    """Evaluación del modelo en test set con matriz de confusión."""
    probs  = np.asarray(result.model.predict(result.params.to_numpy(), exog=X_test.to_numpy()))
    y_pred = np.argmax(probs, axis=1)

    acc            = accuracy_score(y_test_vals, y_pred)
    majority_class = pd.Series(y_test_vals).value_counts().idxmax()
    baseline_acc   = (pd.Series(y_test_vals) == majority_class).mean()

    print(f'\n--- {label} ---')
    print(f'Accuracy en test:           {acc:.4f}')
    print(f'Baseline (clase {majority_class}, mayoritaria): {baseline_acc:.4f}')
    print(f'Ganancia vs. baseline:      {acc - baseline_acc:+.4f}')
    print('\nClassification report:')
    print(classification_report(y_test_vals, y_pred,
                                 target_names=['detractor', 'passive', 'promoter']))

    cm = confusion_matrix(y_test_vals, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['detractor', 'passive', 'promoter'],
                yticklabels=['detractor', 'passive', 'promoter'])
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
    ax.set_title(f'Matriz de confusión — {label}\n(test, n={len(y_test_vals):,})')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}{filename}')
    plt.show()
    return acc, baseline_acc

# Modelo A en test set (ajustado en train en § 4)
acc_A, base_A = confusion_report_test(
    result_train, X_rate_test_c, y_test.values,
    'Modelo A — rate_ (test set)', '08_confusion_test_modeloA.png'
)

## 9. Comprobaciones de robustez: regresión logística binaria

Para validar el **supuesto de líneas paralelas** (odds proporcionales) del modelo ordinal,
se ajustan dos modelos de regresión logística binaria:

- **Comprobación A**: promotor (1) vs. resto (0)
- **Comprobación B**: detractor (1) vs. resto (0)

Si los coeficientes de Comprobación A y Comprobación B son proporcionales entre sí (y de signo coherente
con el modelo ordinal), el supuesto de odds proporcionales queda verificado empíricamente.

In [0]:
# X limpia para regresión binaria (mismas columnas que Modelo A)
X_rate_clean = X_rate[[c for c in rate_cols if c not in high_vif_rate]]

# Check A: promotor vs. resto
y_bin_A = (df['nps_ord'] == 2).astype(int)
lr_A    = LogisticRegression(max_iter=1000, random_state=42)
lr_A.fit(X_rate_clean, y_bin_A)
coef_lr_A = pd.DataFrame({
    'feature':    X_rate_clean.columns,
    'coef_checkA': lr_A.coef_[0]
}).sort_values('coef_checkA', ascending=False).reset_index(drop=True)

# Check B: detractor vs. resto
y_bin_B = (df['nps_ord'] == 0).astype(int)
lr_B    = LogisticRegression(max_iter=1000, random_state=42)
lr_B.fit(X_rate_clean, y_bin_B)
coef_lr_B = pd.DataFrame({
    'feature':     X_rate_clean.columns,
    'coef_checkB': lr_B.coef_[0]
}).sort_values('coef_checkB', ascending=False).reset_index(drop=True)

print('Check A — Promotor vs. resto:')
print(coef_lr_A.to_string(index=False))
print('\nCheck B — Detractor vs. resto:')
print(coef_lr_B.to_string(index=False))

In [0]:
# Comparar signos con el modelo ordinal para verificar odds proporcionales
comparison = (
    coef_A[['feature', 'coef']].rename(columns={'coef': 'coef_ordinal'})
    .merge(coef_lr_A.rename(columns={'coef_checkA': 'coef_promoVsResto'}), on='feature')
    .merge(coef_lr_B.rename(columns={'coef_checkB': 'coef_detractVsResto'}), on='feature')
)
# Consistencia: signo ordinal == signo Check A == -signo Check B
comparison['consist_A'] = (np.sign(comparison['coef_ordinal'])
                            == np.sign(comparison['coef_promoVsResto']))
comparison['consist_B'] = (np.sign(comparison['coef_ordinal'])
                            != np.sign(comparison['coef_detractVsResto']))
comparison['consistente'] = comparison['consist_A'] & comparison['consist_B']

print('=== Verificación del supuesto de líneas paralelas ===')
cols_show = ['feature', 'coef_ordinal', 'coef_promoVsResto', 'coef_detractVsResto', 'consistente']
print(comparison[cols_show].to_string(index=False))

pct = comparison['consistente'].mean() * 100
print(f'\nPorcentaje de variables consistentes: {pct:.1f}%')
print('(100% = supuesto de líneas paralelas perfectamente verificado)')

# Scatter: coef. ordinal vs. coef. de cada check binario
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, xcol, xlab in [
    (axes[0], 'coef_promoVsResto',  'Coef. Check A (promotor vs. resto)'),
    (axes[1], 'coef_detractVsResto','Coef. Check B (detractor vs. resto)')
]:
    ax.scatter(comparison['coef_ordinal'], comparison[xcol], alpha=0.8)
    for _, row in comparison.iterrows():
        ax.annotate(LABEL_MAP.get(row['feature'], row['feature'])[:14],
                    (row['coef_ordinal'], row[xcol]), fontsize=7, alpha=0.7)
    lo = min(comparison[['coef_ordinal', xcol]].min())
    hi = max(comparison[['coef_ordinal', xcol]].max())
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.4, label='y=x')
    ax.axhline(0, color='gray', alpha=0.3); ax.axvline(0, color='gray', alpha=0.3)
    ax.set_xlabel('Coef. Modelo Ordinal A')
    ax.set_ylabel(xlab)
    ax.set_title(xlab)
plt.suptitle('Supuesto de líneas paralelas\n(coherencia coeficientes ordinal vs. binarios)',
             y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}09_parallel_lines_check.png', bbox_inches='tight')
plt.show()

## 10. Exportar resultados

In [0]:
import glob

# Tablas de coeficientes — cuatro modelos
for name, df_out in [
    ('coef_modeloA_rate',          coef_A),
    ('coef_modeloB_intensity',     coef_B),
    ('coef_modeloC_rate_behav',    coef_C),
    ('coef_modeloD_rate_logtotal', coef_D),
]:
    df_out.to_csv(f'{FIGURES_DIR}{name}.csv', index=False)

# Comparación de modelos
df_metrics.to_csv(f'{FIGURES_DIR}comparacion_cuatro_modelos.csv', index=False)

# VIF
vif_rate.to_csv(f'{FIGURES_DIR}vif_rate.csv', index=False)
vif_intens.to_csv(f'{FIGURES_DIR}vif_intensity.csv', index=False)

# Resumen de variables derivadas
feat_summary.to_csv(f'{FIGURES_DIR}feature_summary.csv', index=False)

# Métricas de validación cruzada
if cv_accuracies:
    pd.DataFrame({'fold': range(1, len(cv_accuracies) + 1),
                  'cv_accuracy': cv_accuracies}).to_csv(
        f'{FIGURES_DIR}cv_accuracies_modeloA.csv', index=False)

# Métricas de test set
pd.DataFrame([{
    'modelo': 'A_test',
    'accuracy': acc_A,
    'baseline': base_A,
    'ganancia_vs_baseline': round(acc_A - base_A, 4)
}]).to_csv(f'{FIGURES_DIR}metricas_test_set.csv', index=False)

# Robustez: verificación líneas paralelas
comparison.to_csv(f'{FIGURES_DIR}robustez_lineas_paralelas.csv', index=False)

print('Análisis de regresión completado.')
print(f'Resultados guardados en: {FIGURES_DIR}')
print('\nFicheros generados:')
for f in sorted(glob.glob(f'{FIGURES_DIR}*.csv') + glob.glob(f'{FIGURES_DIR}*.png')):
    print(f'  {os.path.basename(f)}')